In [1]:
import wikipedia

# 검색
print(wikipedia.search("Python programming"))

# 요약 가져오기
#  sentences=1 → 첫 문장만 반환,sentences=2 → 앞에서 2문장 반환, sentences=0 → 전체 요약 반환 (문장 제한 없음)
print(wikipedia.summary("Python (programming language)", sentences=2))

# 언어 변경 (기본: 영어)
wikipedia.set_lang("ko")  # 한국어 위키피디아 사용
print(wikipedia.summary("인공지능", sentences=5))

# 원문 페이지 가져오기
page = wikipedia.page("인공지능")
print('******')
print(page)
print(dir(page))
print('******')
print(page.title)      # 제목
print(page.url)        # URL
print(page.content[:500])  # 본문 일부


['Python (programming language)', 'History of Python', 'Outline of the Python programming language', 'Python syntax and semantics', 'Core Python Programming', 'Mojo (programming language)', 'Zen of Python', 'Guido van Rossum', 'Flask (web framework)', 'Python']
Python is a high-level, general-purpose programming language that emphasizes code readability, simplicity, and ease-of-writing with the use of significant indentation, an extensive ("batteries-included") standard library, and garbage collection. Python supports multiple programming paradigms but with an emphasis on object-oriented programming and dynamic typing.
인공지능(영어: artificial intelligence, AI)은 컴퓨터가 학습, 추론, 지각, 언어 처리, 문제 해결, 계획과 의사 결정처럼 지능과 관련된 일을 수행하도록 하는 방법을 연구하는 컴퓨터 과학 분야이다. 이러한 방법으로 만든 프로그램이나 기계도 인공지능이라고 부른다. 인공지능 시스템은 입력을 처리해 예측값, 추천, 결정, 글·그림 같은 결과를 만들며, 반드시 사람의 사고 과정을 그대로 흉내 내는 것은 아니다.
인공지능에는 사람이 사실과 규칙을 직접 적어 넣는 기호주의 인공지능과, 여러 데이터에서 규칙성을 찾아 모델을 조정하는 기계 학습 등 다양한 접근이 있다. 딥 러닝은 여러 층의 인공 신경망을 사용하는 기계 학습의 한 갈래이고, 생성형 인공

In [2]:
import wikipedia

def wikipedia_search(question):
    wikipedia.set_lang("ko")
    try:
        search_result = wikipedia.search(question)[0]
        print('search_result:', search_result)
        wiki_summary = wikipedia.summary(search_result, sentences=5)
    except Exception as e:
        wiki_summary = "위키피디아에서 정보를 찾을 수 없습니다."
    return {"summary": wiki_summary}

wikipedia_search("세종대왕은 누구야?" )

search_result: 대왕 세종


{'summary': '《대왕 세종(大王世宗)》은 2008년 1월 5일에서부터 2008년 11월 16일까지 방송되었던 KBS 2TV 대하 드라마 작품이다.\n\n\n== 트리비아 ==\n조선 태종 시대부터 세종 시대를 대부분의 배경으로 하고 있다. 제1회 방송 시작 하루 전인 2008년 1월 4일에 제작 뒷이야기를 다룬 스페셜 방송이 있었으며, 2008년 11월 16일까지 총 86부작으로 방송되었다.\n당초 KBS 드라마 PD인 이성주가 담당 PD로 낙점되었으나, KBS 드라마 2팀장직으로 발령되면서 연출자가 바뀌었다.\n아울러 2007년 12월 첫 회가 나갈 예정이었지만 전작 《대조영》의 연장에 따라 2008년 1월로 첫 방송일이 변경되었다.'}

In [3]:
import json
import wikipedia
from openai import OpenAI


def wikipedia_search(question):
    wikipedia.set_lang("ko")
    try:
        search_result = wikipedia.search(question)[0]
        wiki_summary = wikipedia.summary(search_result, sentences=5)
    except Exception as e:
        wiki_summary = "위키피디아에서 정보를 찾을 수 없습니다."
    return {"summary": wiki_summary}


wikifunc = [{
    "type": "function",
    "function": {
        "name": "wikipedia_search",
        "description": "입력된 질문에 대해 필요하다면 위키피디아에서  정보를 검색합니다.",
        "parameters": {
            "type": "object",
            "properties": {
                "question": {
                    "type": "string",
                    "description": " 주제 또는 질문"
                }
            },
            "required": ["question"]
        }
    }
}]


messages = [{"role": "user", "content": "세종대왕에 대해 알려줘"}]

response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=messages,
    tools=wikifunc,
    tool_choice="auto"
)


tool_call = response.choices[0].message.tool_calls[0]
print(tool_call)
args = json.loads(tool_call.function.arguments)
print( args)
result = wikipedia_search(args["question"])
print(result)

# messages = [{"role": "user", "content": "세종대왕에 대해 알려줘"},
#             {"role": "function","name": "wikipedia_search",
#              "content":'{summany:위키피디아답변}'}]

messages.append({
    "role": "function",
    "name": "wikipedia_search",
    "content": json.dumps(result)
})

response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=messages,
    temperature=0
)


print('--------------')
print(response.choices[0].message.content)


NameError: name 'client' is not defined